# TLINK Binary Classification - DistilBERT (Encoder Comparison)

This notebook is a **direct comparison** to the Pythia-70M (decoder) notebook.
Everything is identical except the model. The goal is to see how much difference
the encoder vs decoder architecture makes on a classification task.

---

### Why DistilBERT is better suited for this task

| | Pythia-70M (decoder) | DistilBERT (encoder) |
|---|---|---|
| Reading direction | Left-to-right only | Bidirectional (both directions) |
| Classification token | Last token (hack) | `[CLS]` token (purpose-built) |
| Parameters | 70M | 66M |
| Originally designed for | Text generation | Understanding/classification |
| Padding side needed | Left (special case) | Right (default) |
| Gotchas | 5 workarounds needed | None |

Bidirectional reading matters here because the temporal relationship between
`<e1>` and `<e2>` often depends on words that appear **after** both spans.
A decoder model never sees those words when processing earlier tokens.

---
### Before running: Runtime -> Change runtime type -> T4 GPU -> Save


## Cell 1 - Check GPU


In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO GPU - go to Runtime -> Change runtime type -> T4 GPU')


GPU: Tesla T4
Memory: 15.6 GB


## Cell 2 - Install packages

**After this cell finishes: Runtime -> Restart session, then run from Cell 3 onward.**


In [ ]:
!pip uninstall -q -y transformers tokenizers accelerate
!pip install -q transformers datasets accelerate scikit-learn
print('Done. Now: Runtime -> Restart session, then run from Cell 3.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.0 MB/s eta 0:00:00
Done. Now: Runtime -> Restart session, then run from Cell 3.


## Cell 3 - Imports

Identical imports to the Pythia notebook — the HuggingFace API is the same
regardless of the underlying model architecture.


In [ ]:
import os, time, math
import torch
import numpy as np
from torch import nn
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    classification_report,
)
import transformers
print(f'transformers : {transformers.__version__}')
print(f'torch        : {torch.__version__}')
print(f'GPU          : {torch.cuda.is_available()}')
print('All imports OK')


transformers : 5.0.0
torch        : 2.10.0+cu128
GPU          : True
All imports OK


## Cell 4 - Config

The only meaningful difference from the Pythia config:
- Different `MODEL_NAME`
- `LEARNING_RATE = 2e-5` instead of `2e-6` — DistilBERT is more stable and
  can handle a higher LR without loss exploding, because it was pre-trained
  specifically for fine-tuning on downstream tasks


In [ ]:
MODEL_NAME    = 'distilbert-base-uncased'
DATASET_NAME  = 'mdg-nlp/tlink-extr-classification-sentence-2-label'
OUTPUT_DIR    = './tlink_model_distilbert'
MAX_LENGTH    = 256
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5
EARLY_STOPPING_PATIENCE = 3
LABEL2ID = {'NO': 0, 'YES': 1}
ID2LABEL = {0: 'NO', 1: 'YES'}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Model  : {MODEL_NAME}')
print(f'Device : {DEVICE}, LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}')


Model  : distilbert-base-uncased
Device : cuda, LR: 2e-05, Batch: 32


## Cell 5 - Load dataset


In [ ]:
dataset = load_dataset(DATASET_NAME)
print(f"Train     : {len(dataset['train']):,}")
print(f"Validation: {len(dataset['validation']):,}")
print(f"Test      : {len(dataset['test']):,}")
counts = Counter(dataset['train']['label'])
total  = len(dataset['train'])
print('\nLabel distribution:')
for lbl, cnt in sorted(counts.items()):
    print(f'  {lbl}: {cnt:,} ({cnt/total*100:.1f}%)')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.jsonl:   0%|          | 0.00/18.2M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/59.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61440 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7422 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Train     : 61,440
Validation: 7,422
Test      : 200

Label distribution:
  NO: 5,500 (9.0%)
  YES: 55,940 (91.0%)


## Cell 6 - Tokenizer

Notice how much simpler this is compared to Pythia:
- **No pad token fix needed** — DistilBERT has `[PAD]` built in
- **No padding side fix needed** — DistilBERT reads from `[CLS]` at position 0,
  so right-padding (the default) works perfectly
- We still register the span tags as special tokens — that applies to any model


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Register XML span markers as single tokens
# (same as Pythia — this is dataset-specific, not model-specific)
special_tokens = ['<e1>', '</e1>', '<e2>', '</e2>', '<t1>', '</t1>', '<t2>', '</t2>']
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})

# No padding_side fix needed — right padding is correct for encoder models
print(f'Vocab size   : {len(tokenizer):,}')
print(f'Pad token    : {tokenizer.pad_token}')   # already set to [PAD]
print(f'Padding side : {tokenizer.padding_side}') # already right
print('No workarounds needed for DistilBERT')


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size   : 30,530
Pad token    : [PAD]
Padding side : right
No workarounds needed for DistilBERT


## Cell 7 - Tokenize dataset

Identical to Pythia. The tokenizer API is the same regardless of model type.


In [ ]:
def tokenize(batch):
    encoding = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )
    encoding['labels'] = [LABEL2ID[lbl] for lbl in batch['label']]
    return encoding

tokenized = dataset.map(tokenize, batched=True, batch_size=1000)
tokenized = tokenized.remove_columns(['id', 'doc_id', 'text', 'label'])
tokenized.set_format('torch')
print(f'Columns    : {tokenized["train"].column_names}')
print(f'Train size : {len(tokenized["train"]):,}')


Map:   0%|          | 0/61440 [00:00<?, ? examples/s]

Map:   0%|          | 0/7422 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Columns    : ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
Train size : 61,440


## Cell 8 - Load model

Same API call as Pythia, completely different architecture underneath:

**Pythia:** `tokens -> causal transformer -> last token (512-dim) -> Linear(512->2)`

**DistilBERT:** `[CLS] tokens -> bidirectional transformer -> [CLS] token (768-dim) -> Linear(768->2)`

The `[CLS]` token is prepended to every input and attends to all other tokens
in both directions simultaneously. By the final layer it has aggregated
information from the entire sequence — purpose-built for classification.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
# Resize embeddings to include the 8 new span tag tokens
model.resize_token_embeddings(len(tokenizer))

# Note: no need to set pad_token_id — DistilBERT already knows its pad token
total = sum(p.numel() for p in model.parameters())
print(f'Model      : {MODEL_NAME}')
print(f'Parameters : {total:,}')
print(f'Device     : {DEVICE}')


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `m

Model      : distilbert-base-uncased
Parameters : 66,961,154
Device     : cuda


## Cell 9 - Metrics


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy'  : accuracy_score(labels, preds),
        'f1_yes'    : f1_score(labels, preds, average='binary', pos_label=1),
        'f1_no'     : f1_score(labels, preds, average='binary', pos_label=0),
        'f1_macro'  : f1_score(labels, preds, average='macro'),
        'precision' : precision_score(labels, preds, average='binary', pos_label=1, zero_division=0),
        'recall'    : recall_score(labels, preds, average='binary', pos_label=1, zero_division=0),
    }
print('compute_metrics defined')


compute_metrics defined


## Cell 10 - Class weights and WeightedTrainer

Same weighted loss approach as the Pythia notebook.
The imbalance is a property of the **dataset**, not the model,
so we apply the same fix regardless of architecture.

One difference: DistilBERT runs in float32 by default, so we do **not**
need to call `.half()` on the class weights tensor — unlike Pythia which
ran into the `expected Half but found Float` error.


In [ ]:
no_count    = 5500
yes_count   = 55940
total_count = no_count + yes_count
num_classes = 2

# Square-root scaling — gentler than raw inverse frequency
w_no  = math.sqrt(total_count / (num_classes * no_count))
w_yes = math.sqrt(total_count / (num_classes * yes_count))

# DistilBERT runs in float32 — no .half() needed
class_weights = torch.tensor([w_no, w_yes]).to(DEVICE)
print(f'Class weights: NO={w_no:.3f}, YES={w_yes:.3f}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(weight=class_weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    fp16=False,
    bf16=False,
    dataloader_num_workers=2,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
print('WeightedTrainer configured')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Class weights: NO=2.363, YES=0.741
WeightedTrainer configured


## Cell 11 - Train

Expected on T4 GPU: **3-7 minutes per epoch** — slightly faster than Pythia
because DistilBERT's architecture is more optimised for batched classification.

What to watch for vs Pythia:
- Training loss should decrease **more smoothly** — no oscillation
- `f1_no` should reach meaningful values **earlier** (epoch 1 or 2)
- `f1_macro` should be **higher overall**


In [ ]:
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f'Training complete in {elapsed/60:.1f} minutes')


Epoch,Training Loss,Validation Loss,Accuracy,F1 Yes,F1 No,F1 Macro,Precision,Recall
1,0.250743,0.279983,0.949205,0.972286,0.696213,0.834249,0.962591,0.982177
2,0.246350,0.214349,0.961870,0.979082,0.784791,0.881936,0.974544,0.983663
3,0.148519,0.261960,0.962140,0.979181,0.791388,0.885284,0.976937,0.981435
4,0.102807,0.274344,0.961062,0.978531,0.791034,0.884782,0.978894,0.978167
5,0.100521,0.336877,0.960119,0.978038,0.783309,0.880674,0.977168,0.978910


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete in 120.6 minutes


## Cell 12 - Evaluate

Compare these numbers directly against the Pythia results:

| Metric | Pythia-70M (decoder) | DistilBERT (encoder) |
|---|---|---|
| f1_macro | ~0.50 | ??? |
| f1_no | ~0.23 | ??? |
| f1_yes | ~0.78 | ??? |
| Accuracy | ~0.65 | ??? |


In [ ]:
print('=== Validation set ===')
val_results = trainer.evaluate(tokenized['validation'])
for k, v in val_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

print('\n=== Test set - full classification report ===')
test_out    = trainer.predict(tokenized['test'])
preds       = np.argmax(test_out.predictions, axis=-1)
true_labels = test_out.label_ids
print(classification_report(true_labels, preds, target_names=['NO', 'YES']))


=== Validation set ===


  eval_loss: 0.2620
  eval_accuracy: 0.9621
  eval_f1_yes: 0.9792
  eval_f1_no: 0.7914
  eval_f1_macro: 0.8853
  eval_precision: 0.9769
  eval_recall: 0.9814
  eval_runtime: 62.0854
  eval_samples_per_second: 119.5450
  eval_steps_per_second: 1.8680
  epoch: 5.0000

=== Test set - full classification report ===
              precision    recall  f1-score   support

          NO       0.99      0.71      0.83       100
         YES       0.77      0.99      0.87       100

    accuracy                           0.85       200
   macro avg       0.88      0.85      0.85       200
weighted avg       0.88      0.85      0.85       200



## Cell 13 - Save model


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}  ({size/1e6:.1f} MB)')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./tlink_model_distilbert/
  checkpoint-1920  (0.0 MB)
  checkpoint-3840  (0.0 MB)
  checkpoint-5760  (0.0 MB)
  checkpoint-7680  (0.0 MB)
  checkpoint-9600  (0.0 MB)
  config.json  (0.0 MB)
  model.safetensors  (267.9 MB)
  tokenizer.json  (0.7 MB)
  tokenizer_config.json  (0.0 MB)
  training_args.bin  (0.0 MB)


## Cell 14 - Download model


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('tlink_model_distilbert', 'zip', '.', OUTPUT_DIR)
files.download('tlink_model_distilbert.zip')


NameError: name 'OUTPUT_DIR' is not defined

## Cell 15 - Inference demo

Same four examples used in the Pythia notebook for a direct comparison.

Pythia-70M results for reference:
```
The department said Wednesday...   YES  -> YES  0.927  OK
The marine was sentenced...        NO   -> YES  0.682  WRONG
She arrived before noon...         YES  -> YES  0.840  OK
The vote was counted after...      YES  -> YES  0.941  OK
```


In [ ]:
from transformers import pipeline
clf = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

examples = [
    ('The department <e1>said</e1> <t1>Wednesday</t1> the US filed a note.', 'YES'),
    ('The marine was <e1>sentenced</e1> for <e2>raping</e2> a woman.',        'NO'),
    ('She <e1>arrived</e1> at the office <t1>before noon</t1>.',              'YES'),
    ('The vote was <e1>counted</e1> <t1>after</t1> polls <e2>closed</e2>.',  'YES'),
]

print('DistilBERT results:')
print(f"{'Text':<58} {'Expected':<10} {'Got':<6} {'Conf'}")
print('-' * 82)
correct = 0
for text, expected in examples:
    r = clf(text, truncation=True, max_length=MAX_LENGTH)[0]
    mark = 'OK' if r['label'] == expected else 'WRONG'
    correct += int(r['label'] == expected)
    print(f"{text[:56]:<58} {expected:<10} {r['label']:<6} {r['score']:.3f}  {mark}")
print(f'\nDemo accuracy: {correct}/{len(examples)}')
print('\n(Compare to Pythia: 3/4)')


DistilBERT results:
Text                                                       Expected   Got    Conf
----------------------------------------------------------------------------------
The department <e1>said</e1> <t1>Wednesday</t1> the US f   YES        YES    0.996  OK
The marine was <e1>sentenced</e1> for <e2>raping</e2> a    NO         YES    0.967  WRONG
She <e1>arrived</e1> at the office <t1>before noon</t1>.   YES        YES    0.992  OK
The vote was <e1>counted</e1> <t1>after</t1> polls <e2>c   YES        YES    0.992  OK

Demo accuracy: 3/4

(Compare to Pythia: 3/4)
